# 第6章 主成分分析及Python计算

In [ ]:
#%cd "[本地路径已移除]
%run init.py

## 6.1 主成分分析的概念

### 6.1.1 主成分分析的提出

### 6.1.2 主成分的直观解释

$\begin{cases}
y_1=\cos \theta x_1+\sin \theta x_2 \\
y_2=-\sin \theta x_1+\cos \theta x_2
\end{cases}$

In [ ]:
import matplotlib.pyplot as plt              #加载基本绘图包
import pandas as pd                         #加载数据处理包
plt.rcParams['font.sans-serif']=['Heiti TC'];  #设置中文字体为黑体

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
x1=[147,171,175,159,155,152,158,154,164,168,166,159,164,177]  #身高
x2=[32,57,64,41,38,35,44,41,54,57,49,47,46,63]                #体重

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(x1, x2); #绘制散点图
plt.xlabel('x1');plt.ylabel('x2');      

In [ ]:
from matplotlib.patches import Ellipse
fig=plt.figure();
ax=fig.add_subplot(111)
ell1=Ellipse(xy=(162,48),width=48,height=8,angle=48,facecolor='yellow',alpha=0.3) 
ax.add_patch(ell1) #绘制椭圆 
plt.scatter(x1, x2);plt.xlabel('x1');plt.ylabel('x2')    
plt.plot([146,178],[30,66]);plt.plot([162,166],[54,47]); #绘制线段
plt.text(178,66,'y1');plt.text(161,55,'y2');

## 6.2 主成分分析的性质

### 6.2.1 主成分的说明

In [ ]:
import pandas as pd
pd.set_option('display.precision',4)  #数据框输出精度
X=pd.DataFrame({'x1':x1,'x2':x2});#X  #构建数据框 X

In [ ]:
S=X.cov();S    #协方差阵

In [ ]:
R=X.corr();R  #相关系数阵

In [ ]:
# 1. 数据标准化（均值为0，方差为1）
X_mean = np.mean(X, axis=0)
x_var = np.var(X, axis=0)
X_centered = (X - X_mean) / np.sqrt(x_var)


In [ ]:
# 2. 计算协方差矩阵
cov_matrix = np.cov(X_centered, rowvar=False)
cov_matrix

In [ ]:
# 3. 计算协方差矩阵的特征值和特征向量
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
eigenvalues, eigenvectors

In [ ]:
# 5. 选择前两个主成分
n_components = 2
selected_eigenvectors = eigenvectors[:, :n_components]


### 6.2.2 主成分的推导

In [ ]:
import numpy as np  #from numpy.linalg import svd
np.set_printoptions(4)
d1,u1,v1=np.linalg.svd(S) #协差阵的奇异值分解 S=UDV'
print('d1:\n',d1,'\n','u1:\n',u1,'\n','v1:\n',v1)

In [ ]:
d2,u2,v2=np.linalg.svd(R)  #相关阵的奇异值分解 R=UDV'
print('d1:\n',d2,'\n','u1:\n',u2,'\n','v1:\n',v2)

In [ ]:
# 1. 计算协方差矩阵的特征值和特征向量
eigenvalues, eigenvectors = np.linalg.eig(S)
eigenvalues, eigenvectors

In [ ]:
# 计算原始数据的方差（对角线元素）
sigma_jj = np.var(X, axis=0, ddof=1)  # 样本方差

In [ ]:
(np.sqrt(eigenvalues) * X ) / (sigma_jj ** 0.5)

In [ ]:
# 计算主成分系数矩阵 A
A = np.zeros_like(eigenvectors)
for i in range(len(eigenvectors)):  # 遍历每个主成分
    A[:, i] = (np.sqrt(eigenvalues[i]) * eigenvectors[:, i]) / np.sqrt(sigma_jj)
A


In [ ]:
np.dot(X, A)

### 6.2.3 主成分的计算

1.主成分方差

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit(X)         #拟合主成分
Vi=pca.explained_variance_;Vi             #主成分方差
#pd.DataFrame(pca.explained_variance_,index=['PC1','PC2'])   #主成分方差

2.主成分系数

In [ ]:
#DataFrame(pca.components_,columns=['PC1','PC2'])        #主成分负荷
pca.components_     #主成分负荷

In [ ]:
# 将主成分得分转换为 DataFrame
scores = pd.DataFrame(pca.components_, index=X.columns, columns=['PC1', 'PC2'])
print("主成分得分:\n", scores)

3. 主成分系数及主成分得分的  <big><mark>**分步计算**<mark></big>

In [ ]:
# 数据中心化
x_var = np.var(X, axis=0)
X_centered = (X - X_mean)
print("中心化后的数据:\n", X_centered)

In [ ]:
# 计算协方差矩阵
cov_matrix = np.cov(X_centered, rowvar=False)
print("协方差矩阵:\n", cov_matrix)

In [ ]:
# 计算协方差矩阵的特征值和特征向量
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
print("特征值:\n", eigenvalues)
print("特征向量:\n", eigenvectors)


In [ ]:
# 将特征值从大到小排序
sorted_indices = np.argsort(eigenvalues)[::-1]
sorted_eigenvalues = eigenvalues[sorted_indices]
sorted_eigenvectors = eigenvectors[:, sorted_indices]

print("排序后的特征值:\n", sorted_eigenvalues)
print("排序后的特征向量:\n", sorted_eigenvectors)

In [ ]:
# 选择前两个主成分
n_components = 2
selected_eigenvectors = sorted_eigenvectors[:, :n_components]
print("选择的主成分:\n", selected_eigenvectors)

In [ ]:
# 将数据投影到主成分上 （未排序）
X_pca = np.dot(X_centered, eigenvectors)
print("投影后的数据:\n", X_pca)

In [ ]:
# 将数据投影到主成分上,即计算主成分得分
X_pca = np.dot(X_centered, selected_eigenvectors)
X_pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
print("投影后的数据:\n", X_pca_df)

In [ ]:
X_pca_df.plot.scatter(x='PC1', y='PC2')
plt.axhline(y=0, color='blue', linestyle=':')
plt.axvline(x=0, color='blue', linestyle='--')

3.主成分得分

In [ ]:
scores=pd.DataFrame(pca.fit_transform(X),columns=['PC1','PC2']);
scores #主成分得分
#scores=pca.fit_transform(X);scores #主成分得分

In [ ]:
scores.plot('PC1','PC2',kind='scatter')           #绘制主成分得分散点图
plt.axhline(y=0,ls=":");plt.axvline(x=0,ls=":"); #添加水平和垂直直线

In [ ]:
scores.corr().round(4)                  #主成分得分相关阵

In [ ]:
scores.cov().round(4)                    #主成分得分协方差阵

4.方差贡献率

In [ ]:
Wi=pca.explained_variance_ratio_;Wi   #方差贡献率 

In [ ]:
Wi.sum()             #方差累计贡献率

In [ ]:
# 计算总方差
total_variance = np.sum(sorted_eigenvalues)
print("总方差:", total_variance)

# 计算方差贡献率
explained_variance_ratio = sorted_eigenvalues / total_variance
print("方差贡献率:\n", explained_variance_ratio)
print('方差累积贡献率:\n',explained_variance_ratio.sum())

## 6.3 主成分分析步骤
### 6.3.1 计算过程

In [ ]:
import numpy as np
def PCscores(X,m=2): #主成分评价函数
    from sklearn.decomposition import PCA
    Z=(X-X.mean())/X.std() #数据标准化
    p=Z.shape[1]
    pca = PCA(n_components=p).fit(Z)
    Vi=pca.explained_variance_;Vi   # 获取特征值(各主成分的方差)
    Wi=pca.explained_variance_ratio_;Wi # 获取各主成分的方差贡献率
    
    # 创建方差相关的DataFrame
    Vars=pd.DataFrame({'Variances':Vi});Vars  #,index=X.columns
    Vars.index=['Comp%d' %(i+1) for i in range(p)]
    Vars['Explained']=Wi*100;Vars
    Vars['Cumulative']=np.cumsum(Wi)*100;
    print("\n方差贡献:\n",round(Vars,4))
    
    # 提取前m个主成分的负荷矩阵
    Compi=['Comp%d' %(i+1) for i in range(m)]
    loadings=pd.DataFrame(pca.components_[:m].T,columns=Compi,index=X.columns);
    print("\n主成分负荷:\n",round(loadings,4))
    
    # 计算主成分得分
    scores=pd.DataFrame(pca.fit_transform(Z)).iloc[:,:m];
    scores.index=X.index; 
    scores.columns=Compi;scores
    
    # 计算综合得分和排名
    scores['Comp']=scores.dot(Wi[:m]);scores
    scores['Rank']=scores.Comp.rank(ascending=False).astype(int);
    return scores #print('\n综合得分与排名:\n',round(scores,4))

### 6.3.2 实证分析

In [ ]:
d31=pd.read_excel('mvsData.xlsx','d31',index_col=0)
d31_pcs=PCscores(d31);print(d31_pcs)

In [ ]:
d31=pd.read_excel('mvsData.xlsx','d31',index_col=0); d31

In [ ]:
d31_pcs=PCscores(d31);
#print(d31_pcs)

In [ ]:
print(d31_pcs.sort_values('Rank'))

In [ ]:
import matplotlib.pyplot as plt            
plt.rcParams['font.sans-serif']=['Heiti TC'];  #中文黑体SimHei
plt.rcParams['axes.unicode_minus']=False; #正常显示图中正负号
def Scoreplot(Scores): #自定得分图绘制函数
    plt.plot(Scores.iloc[:,0],Scores.iloc[:,1],'*'); 
    plt.xlabel(Scores.columns[0]);plt.ylabel(Scores.columns[1])
    plt.axhline(y=0,ls=':');plt.axvline(x=0,ls=':')
    for i in range(len(Scores)):
        plt.text(Scores.iloc[i,0],Scores.iloc[i,1],Scores.index[i])

In [ ]:
Scoreplot(d31_pcs)

## 6.4 主成分分析注意事项

## 案例6 电信业发展的主成分分析

In [ ]:
Case6=pd.read_excel('mvsCase.xlsx','Case6',index_col=0); #Case6

In [ ]:
plt.figure(figsize=(9,5))
import scipy.cluster.hierarchy as sch
Z=(Case6-Case6.mean())/Case6.std() #数据标准化
D=sch.distance.pdist(Z)
H=sch.linkage(D,method='complete');
sch.dendrogram(H,labels=Case6.index); #绘制系统聚类图

In [ ]:
level4 = pd.DataFrame(sch.cut_tree(H), index= Case6.index)[17]+1
level3 = pd.DataFrame(sch.cut_tree(H), index= Case6.index)[18]+1
level2 = pd.DataFrame(sch.cut_tree(H), index= Case6.index)[19]+1

class_41 = level4[level4 == 1].index.tolist()
class_42 = level4[level4 == 2].index.tolist()
class_43 = level4[level4 == 3].index.tolist()
class_44 = level4[level4 == 4].index.tolist()

# 创建一个字典来存储分类结果
data4 = {
    '第1类': [class_41],
    '第2类': [class_42],
    '第3类': [class_43],
    '第4类': [class_44]
}
# 将字典转换为 DataFrame
class4_df = pd.DataFrame(data4)
class4_df

class_31 = level3[level3 == 1].index.tolist()
class_32 = level3[level3 == 2].index.tolist()
class_33 = level3[level3 == 3].index.tolist()

# 创建一个字典来存储分类结果
data3 = {
    '第1类': [class_31],
    '第2类': [class_32],
    '第3类': [class_33]
}
# 将字典转换为 DataFrame
class3_df = pd.DataFrame(data3)

class_21 = level2[level2 == 1].index.tolist()
class_22 = level2[level2 == 2].index.tolist()
# 创建一个字典来存储分类结果
data2 = {
    '第1类': [class_21],
    '第2类': [class_22]
}
# 将字典转换为 DataFrame
class2_df = pd.DataFrame(data2)

In [ ]:
print("分2类：\n",class2_df.T)
print("分3类：\n",class3_df.T)
print("分4类：\n",class4_df.T)

In [ ]:
Case6_pcs=PCscores(Case6);Case6_pcs    #主成分分析

In [ ]:
Scoreplot(Case6_pcs)